<a href="https://colab.research.google.com/github/nev12/gym-dropout-prediction/blob/main/01_data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gym Data Generation

Generating a synthetic gym activity dataset designed to simulate somewhat realistic attendance patterns and churn behaviour.

It is composed out of two csv:
- users data
- checkins data



In [6]:
import numpy as np
import pandas as pd
from datetime import timedelta

In [7]:
np.random.seed(42)

N_USERS = 5000
START_DATE = pd.Timestamp("2025-01-01")
END_DATE = pd.Timestamp("2025-12-31")

user_types = {
    "consistent": {"mean_gap": 3, "churn_prob": 0.05},
    "casual": {"mean_gap": 7, "churn_prob": 0.10},
    "sporadic": {"mean_gap": 14, "churn_prob": 0.20},
    "at_risk": {"mean_gap": 21, "churn_prob": 0.45},
}

workout_types = ["Strength", "HIIT", "Cardio", "Yoga"]
gyms = [f"gym_{i}" for i in range(1, 11)]

## Generate Users data

Holds all the user ids, their types, and signup dates.


In [8]:
users = pd.DataFrame({
    "user_id": [f"user_{i}" for i in range(1, N_USERS+1)],
    "user_type": np.random.choice(
        list(user_types.keys()),
        size = N_USERS,
        p = [0.3, 0.35, 0.2, 0.15]
    )
})

users["signup_date"] = START_DATE + pd.to_timedelta(
    np.random.randint(0, 60, size=N_USERS), unit="D"
)

## Generate Checkins data

Here, all the checkins of each user are stored, with exact checkin and checkout time, gym id, workout type and the number of burned calories.

In [9]:
checkins = []

for _, row in users.iterrows():
  user_id = row["user_id"]
  utype = row["user_type"]
  params = user_types[utype]
  total_checkins = 0

  current_time = row["signup_date"]
  churned = False

  user_gyms = np.random.choice(gyms, size=np.random.randint(1,4), replace=False)
  user_workouts = np.random.choice(workout_types, size=np.random.randint(1,3), replace=False)

  while current_time < END_DATE and not churned:
    gap = np.random.exponential(params["mean_gap"])
    current_time += timedelta(days=gap)

    if current_time >= END_DATE:
      break

    if total_checkins >= 60 :
      if np.random.rand() < params["churn_prob"]:
        churned = True
        break

    duration = np.random.normal(90, 20)
    duration = max(30, min(duration, 180))

    checkins.append({
        "user_id": user_id,
        "gym_id": np.random.choice(user_gyms),
        "checkin_time": current_time,
        "checkout_time": current_time + timedelta(minutes=duration),
        "workout_type": np.random.choice(user_workouts),
        "calories_burned": int(duration * np.random.uniform(6,10))

    })
    total_checkins += 1

checkins = pd.DataFrame(checkins)

## Save

Saving them as csv's.

In [10]:
users.to_csv("users.csv", index=False)
checkins.to_csv("checkins.csv", index=False)